In [2]:
from google.colab import files
uploaded = files.upload()

Saving Book1.xlsx to Book1.xlsx


In [13]:
import pandas as pd

# Get uploaded file name
file_name = list(uploaded.keys())[0]

# Read Excel
df = pd.read_excel(file_name)

# Make first row as header
df.columns = df.iloc[0]

# Remove first row from data
df = df[1:]

# Reset index
df.reset_index(drop=True, inplace=True)

# Clean column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Preview cleaned data
print(df.head())

# Check all columns
print(df.columns.tolist())

0 material_document         posting_date document_header_text reference  \
0        5000239788  2025-01-02 00:00:00         491514856317       997   
1        5000239789  2025-01-02 00:00:00         451514907837      1059   
2        5000239800  2025-01-02 00:00:00         451514907837      1059   
3        5000239800  2025-01-02 00:00:00         451514907837      1059   
4        5000239801  2025-01-02 00:00:00         451514907837      1059   

0 document_date           entry_date movement_type material_type  \
0         45657  2025-01-02 00:00:00           101          VERP   
1         45657                45659           101          VERP   
2         45657                45659           101          VERP   
3         45657                45659           101          VERP   
4         45657                45659           101          VERP   

0 material_document_year       batch  ... accepted_quantity net_order_price  \
0                   2025  0000669755  ...               NOS  

In [5]:
new_columns = [
    'material_document',
    'posting_date',
    'document_header_text',
    'reference',
    'document_date',
    'entry_date',
    'movement_type',
    'material_type',
    'material_document_year',
    'batch',
    'plant',
    'has_reversal_movement_type',
    'supplier',
    'supplier_name',
    'purchase_order',
    'material',
    'material_description',
    'hsn',
    'gst',
    'category',
    'customer_category',
    'material_group',
    'material_group_description',
    'valuation_type',
    'base_unit_of_measure',
    'currency',
    'profit_center',
    'profit_center_description',
    'user_name',
    'time_of_entry',
    'storage_location',
    'price_unit',
    'receipt_amount',
    'receipt_amount_currency',
    'receipt_quantity',
    'receipt_quantity_unit',
    'blocked_stock',
    'blocked_stock_unit',
    'accepted_quantity',
    'accepted_quantity_unit',
    'net_order_price',
    'net_order_price_currency',
    'rejected_qty',
    'rejected_qty_unit',
    'delivery_note_quantity',
    'delivery_note_quantity_unit',
    'final_net_price',
    'total_value_as_per_accept_qty',
    'onboarding_status'
]

df.columns = new_columns

In [6]:
numeric_cols = [
    'receipt_amount',
    'receipt_quantity',
    'blocked_stock',
    'accepted_quantity',
    'net_order_price',
    'rejected_qty',
    'delivery_note_quantity',
    'final_net_price',
    'total_value_as_per_accept_qty'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [7]:
df['posting_date'] = pd.to_datetime(df['posting_date'])

df['month'] = df['posting_date'].dt.to_period('M')

In [8]:
part_frequency = (
    df.groupby('material_description')
    .size()
    .reset_index(name='frequency')
)

conditions = [
    part_frequency['frequency'] >= 20,
    part_frequency['frequency'].between(5, 19),
    part_frequency['frequency'] < 5
]

labels = ['Runner', 'Repeater', 'Stranger']

part_frequency['classification'] = np.select(
    conditions,
    labels,
    default='Unknown'
)

print(part_frequency.head())

                material_description  frequency classification
0                  SHAFT L178.5 CRNG         35         Runner
1           SHAFT L185 LESCO BLENDER          2       Stranger
2   SHAFT L185 LESCO BLENDER 3 SPEED          2       Stranger
3               UNIT BOX ISI STICKER          3       Stranger
4       0.5 MM HYLUM WASHER ID (8.4)         30         Runner


In [9]:
fragmentation = (
    df.groupby('category')['supplier_name']
    .nunique()
    .reset_index(name='unique_suppliers')
    .sort_values(by='unique_suppliers', ascending=False)
)

print(fragmentation.head(10))

                   category  unique_suppliers
0                         0                56
8   Electricals/Electronics                30
25        Special Additives                21
11                Machining                20
28             WIRE HARNESS                17
13                Packaging                15
9                 Fasteners                11
15         Plastic/Polymers                 9
27                     Tape                 8
1                   BEARING                 8


In [10]:
supplier_spend = (
    df.groupby(['category', 'supplier_name'])['total_value_as_per_accept_qty']
    .sum()
    .reset_index()
)

category_total = supplier_spend.groupby('category')[
    'total_value_as_per_accept_qty'
].transform('sum')

supplier_spend['spend_share'] = (
    supplier_spend['total_value_as_per_accept_qty'] / category_total
)

high_dependency = supplier_spend[
    supplier_spend['spend_share'] > 0.7
]

print(high_dependency)

             category                                      supplier_name  \
58            BEARING                       JOY EXPORTS/110002 NEW DELHI   
64        Brass Parts                ADITYA ENGINEERING/110094 NEW DELHI   
69          CAPACITOR  GLOBE CAPACITORS PRIVATE LIMITED/121004 FARIDABAD   
75           Circlips                ADITYA ENGINEERING/110094 NEW DELHI   
125             Foams  COSMOS TAPES & LABELS PVT. LTD./201306 GAUTAMB...   
148           POLYBAG     GLOBAL DYNAMIC INDUSTRIES/203207 GREATER NOIDA   
178  Plastic/Polymers                   P.K.ENTERPRISES/201001 GHAZIABAD   
183       Proprietary                ADITYA ENGINEERING/110094 NEW DELHI   
192          Radiator  CLIFFTON PACKAGINGS PRIVATE LIMITED/201305 GAU...   
198            SOLDER   DXL ELECTRONIC MATERIALS INDIA PVT./201304 NOIDA   
199            SPRING                 EAGLE SPRINGS (INDIA)/110032 DELHI   
201     Safty Harness  SHRI SAI ELECTROWORKS PRIVATE LIMIT/201007 GHA...   
202      She

In [11]:
monthly_spend = (
    df.groupby('month')['total_value_as_per_accept_qty']
    .sum()
    .reset_index()
)

print(monthly_spend)

      month  total_value_as_per_accept_qty
0   2025-01                   8.864896e+07
1   2025-02                   7.770438e+07
2   2025-03                   6.803971e+07
3   2025-04                   7.809733e+07
4   2025-05                   6.882356e+07
5   2025-06                   7.298764e+07
6   2025-07                   1.014167e+08
7   2025-08                   9.569477e+07
8   2025-09                   1.049984e+08
9   2025-10                   7.246181e+07
10  2025-11                   8.069643e+07
11  2025-12                   8.881554e+07


In [12]:
price_variation = (
    df.groupby('material_description')['net_order_price']
    .agg(['mean', 'std', 'max', 'min'])
    .reset_index()
)

price_variation['variation_percent'] = (
    (price_variation['std'] / price_variation['mean']) * 100
)

high_variation = price_variation[
    price_variation['variation_percent'] > 30
]

print(high_variation.head())

                         material_description         mean          std  \
108           5 STAR 6.5 RATING LABEL (PAPER)     0.880952     0.384212   
110         5 STAR 6.6 RATING AIROJEWEL SMART     1.312500     0.983343   
192            ASIAN EPOXY PRIMER GREY 20 LTR  2053.333333  3071.503432   
244  BLACK MULTI STRAND CU WIRE 2.5sqmm-L-550    17.005000     5.296230   
286                              BLADE BOX-X5     6.076429     2.233182   

         max     min  variation_percent  
108     2.50    0.60          43.613282  
110     2.50    0.60          74.921404  
192  5600.00  280.00         149.586206  
244    20.75   13.26          31.145133  
286    12.68    5.23          36.751554  


In [14]:
with pd.ExcelWriter('procurement_insights.xlsx', engine='openpyxl') as writer:

    # Original cleaned data
    df.to_excel(writer, sheet_name='Cleaned_Data', index=False)

    # Runner Repeater Stranger
    part_frequency.to_excel(writer, sheet_name='Part_Classification', index=False)

    # Supplier Fragmentation
    fragmentation.to_excel(writer, sheet_name='Supplier_Fragmentation', index=False)

    # Supplier Dependency
    high_dependency.to_excel(writer, sheet_name='Supplier_Dependency', index=False)

    # Monthly Spend
    monthly_spend.to_excel(writer, sheet_name='Monthly_Trends', index=False)

    # Pricing Anomalies
    high_variation.to_excel(writer, sheet_name='Price_Anomalies', index=False)

print("Excel file exported successfully!")

Excel file exported successfully!
